# Bidease Reporting — demo

Notebook для работы с функциями из `bidease.py`:

1. `get_campaign_dict()` — справочник кампаний из группировок отчёта
2. `get_campaigns_daily_stat(date_from, date_to)` — дневная статистика по кампаниям
3. `get_creatives_daily_stat(date_from, date_to)` — дневная статистика по креативам
4. `get_admin_audit(date_from, date_to)` — сводный аудит по дням (агрегат)

## Перед запуском
1. Установи зависимости (ячейка ниже)
2. Добавь `API_TOKEN` в файл `.env`

## 1. Установка зависимостей (один раз)

In [ ]:
%pip install -q -r requirements.txt

## 2. Импорты и загрузка учётных данных

In [ ]:
import logging
import os
from dotenv import load_dotenv

load_dotenv(".env")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

assert os.environ.get("API_TOKEN"), "API_TOKEN отсутствует в .env"

print("Учётные данные загружены")

In [ ]:
import sys
sys.path.insert(0, ".")

from bidease import (
    get_campaign_dict,
    get_campaigns_daily_stat,
    get_creatives_daily_stat,
    get_admin_audit,
)

## 3. Параметры периода

Обе даты — **включительно** (эксклюзивность `todate` API библиотека учитывает сама).
Все даты — в таймзоне токена (по умолчанию UTC+0).

In [ ]:
from datetime import date, timedelta

DATE_TO   = (date.today() - timedelta(days=1)).isoformat()
DATE_FROM = (date.today() - timedelta(days=7)).isoformat()

# Переопредели вручную если нужно:
# DATE_FROM = "2026-07-01"
# DATE_TO   = "2026-07-14"

print(f"Период: {DATE_FROM} → {DATE_TO}")

## 4. Справочник кампаний

In [ ]:
get_campaign_dict_result = get_campaign_dict()
print(f"Строк: {len(get_campaign_dict_result)}")
get_campaign_dict_result.head(10)

In [ ]:
# Быстрый анализ
if not get_campaign_dict_result.empty:
    print(f"Всего записей: {len(get_campaign_dict_result)}")

## 5. Дневная статистика по кампаниям

Один GET-запрос на весь период (`group=Day+CampaignID`) — без перебора кампаний.

In [ ]:
get_campaigns_daily_stat_result = get_campaigns_daily_stat(DATE_FROM, DATE_TO)
print(f"Строк: {len(get_campaigns_daily_stat_result)}")
get_campaigns_daily_stat_result.head(10)

In [ ]:
# Сводка по дням
if not get_campaigns_daily_stat_result.empty:
    summary = get_campaigns_daily_stat_result.groupby("date")[
        ["impressions", "clicks", "costs_nds", "costs_without_nds", "costs_nds_ak", "costs_without_nds_ak"]
    ].sum()
    display(summary)

## 6. Дневная статистика по креативам

Один GET-запрос на весь период (`group=Day+CampaignID+CreativeID`).

In [ ]:
get_creatives_daily_stat_result = get_creatives_daily_stat(DATE_FROM, DATE_TO)
print(f"Строк: {len(get_creatives_daily_stat_result)}")
get_creatives_daily_stat_result.head(10)

In [ ]:
# Сводка по дням
if not get_creatives_daily_stat_result.empty:
    summary = get_creatives_daily_stat_result.groupby("date")[
        ["impressions", "clicks", "costs_nds", "costs_without_nds"]
    ].sum()
    display(summary)

## 7. Сводный аудит (admin_audit)

Собственного запроса к API нет — агрегат поверх статистики кампаний.

In [ ]:
get_admin_audit_result = get_admin_audit(DATE_FROM, DATE_TO)
print(f"Строк: {len(get_admin_audit_result)}")
get_admin_audit_result.head(10)

## Сохранение в CSV (опционально)

In [ ]:
from datetime import date
from pathlib import Path

today = date.today().isoformat()

# имена файлов содержат дату/период — старые выгрузки сами не перезаписываются, удаляем
for fn_name in [
    "get_campaign_dict",
    "get_campaigns_daily_stat",
    "get_creatives_daily_stat",
    "get_admin_audit",
]:
    for old in Path(".").glob(f"{fn_name}_*.csv"):
        old.unlink()

get_campaign_dict_result.to_csv(f"get_campaign_dict_{today}.csv", index=False, encoding="cp1251", errors="replace")
get_campaigns_daily_stat_result.to_csv(f"get_campaigns_daily_stat_{DATE_FROM}_{DATE_TO}.csv", index=False, encoding="cp1251", errors="replace")
get_creatives_daily_stat_result.to_csv(f"get_creatives_daily_stat_{DATE_FROM}_{DATE_TO}.csv", index=False, encoding="cp1251", errors="replace")
get_admin_audit_result.to_csv(f"get_admin_audit_{DATE_FROM}_{DATE_TO}.csv", index=False, encoding="cp1251", errors="replace")

print("Файлы сохранены")